# Reducing data-cubes using geometry objects

In [ ]:
from earthkit import data as ekd
from earthkit import plots as ekp
from earthkit import transforms as ekt
from earthkit.transforms._tools import earthkit_remote_test_data_file

## Load some test data

All `earthkit-transforms` methods can be called with `earthkit-data` objects (Readers and Wrappers) or with the 
pre-loaded `xarray` or `geopandas` objects.

In this example we will use hourly ERA5 2m temperature data on a 0.5x0.5 spatial grid for the year 2015 as
our physical data; and we will use the NUTS geometries which are stored in a geojson file.

First we lazily load the ERA5 data  and NUTS geometries from our test-data repository.

Note the data is only downloaded when
we use it, e.g. at the `.to_xarray` line, additionally, the download is cached so the next time you run this
cell you will not need to re-download the file (unless it has been a very long time since you have run the
code, please see tutorials in `earthkit-data` for more details in cache management).

In [ ]:
# Get some demonstration ERA5 data, this could be any url or path to an ERA5 grib or netCDF file.
# remote_era5_file = earthkit_remote_test_data_file("era5_temperature_europe_2015.grib") # Large file
remote_era5_file = earthkit_remote_test_data_file("era5_temperature_europe_20150101.grib")
era5_data = ekd.from_source("url", remote_era5_file)
era5_xr = era5_data.to_xarray(time_dims=["valid_time"]).rename({"2t": "t2m"})
era5_xr

In [ ]:
era5_point_cloud = era5_xr.stack(index=("latitude", "longitude"))
era5_point_cloud.latitude.attrs.update({'axis': 'y'})
era5_point_cloud.longitude.attrs.update({'axis': 'x'})
era5_point_cloud


The stacked dataset has a single spatial dimension `index` whose coordinates are the
`(latitude, longitude)` pairs from the original grid.
By setting the `axis` attribute on those coordinates, `earthkit-transforms` can
auto-detect them even though they are no longer the dimension names.


In [ ]:
# Use some demonstration polygons stored, this could be any url or path to geojson file
remote_nuts_url = earthkit_remote_test_data_file("NUTS_RG_60M_2021_4326_LEVL_0.geojson")
nuts_data = ekd.from_source("url", remote_nuts_url).to_geopandas()

nuts_data[:5]

## Reduce data

### Default behaviour

### Reduce over spatial dimension only

`ekt.spatial.reduce` detects the unstructured layout from the `axis` attributes set above and
uses `mask_contains_points` (shapely-based) rather than the rasterize path.

The result has a new `NAME_LATN` dimension (one entry per NUTS region) while the `index` dimension
is reduced away.  The `valid_time` dimension is preserved — each region retains its full time series.


In [ ]:
reduced_data = ekt.spatial.reduce(
    era5_point_cloud,
    nuts_data,
    lat_key="latitude",
    lon_key="longitude",
    all_touched=True,
    mask_dim="NAME_LATN",
)
reduced_data

### Reduce over spatial and time dimensions together

Pass `extra_reduce_dims="valid_time"` to collapse both the spatial `index` dimension and the
time dimension in a single step, returning a scalar per NUTS region.


In [ ]:
reduced_data_collapsed = ekt.spatial.reduce(
    era5_point_cloud,
    nuts_data,
    lat_key="latitude",
    lon_key="longitude",
    extra_reduce_dims="valid_time",
    mask_dim="NAME_LATN",
)
reduced_data_collapsed


### Weighted reduction

Pass `weights="latitude"` to apply cosine-latitude weighting when computing the mean.
This is identical to the regular-grid workflow — the `latitude` coordinate on the `index`
dimension is used to derive the weights.


In [ ]:
reduced_data_weighted = ekt.spatial.reduce(
    era5_point_cloud,
    nuts_data,
    lat_key="latitude",
    lon_key="longitude",
    weights="latitude",
    extra_reduce_dims="valid_time",
    mask_dim="NAME_LATN",
)
reduced_data_weighted


### Visualise: compare unweighted vs latitude-weighted mean

We join the two scalar results back onto the NUTS GeoDataFrame and plot a choropleth
showing the difference between the unweighted and weighted country-mean temperatures.


In [ ]:
import pandas as pd

# Build a GeoDataFrame indexed by country name for easy joining
nuts_indexed = nuts_data.set_index("NAME_LATN")

# Convert both reduced results to pandas and join onto the geometry
df_unweighted = reduced_data_collapsed["t2m"].to_dataframe().rename(columns={"t2m": "t2m_unweighted"})
df_weighted = reduced_data_weighted["t2m"].to_dataframe().rename(columns={"t2m": "t2m_weighted"})

plot_gdf = nuts_indexed.join(df_unweighted).join(df_weighted)
plot_gdf["diff_K"] = plot_gdf["t2m_weighted"] - plot_gdf["t2m_unweighted"]
plot_gdf[["t2m_unweighted", "t2m_weighted", "diff_K"]].head()


In [ ]:
fig = ekp.Figure()
fig.add_map()

# Plot the temperature difference (weighted minus unweighted) as a choropleth
fig.subplot(1).plot(plot_gdf, column="diff_K", style="colormap")
fig.subplot(1).style(title="Weighted − unweighted mean 2 m temperature (K)\npoint-cloud ERA5, 1 Jan 2015")
fig
